In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import lu_factor, lu_solve, lu

# Load the dataset
df = pd.read_csv('dataset/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [16]:
# Sparse matrix
R = df.pivot(index='user_id', columns='item_id', values='rating').to_numpy()
R = np.nan_to_num(R) 
R

array([[5., 3., 4., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.]], shape=(943, 1682))

In [17]:
# Train-test split
def train_test_split(R, test_ratio=0.2, seed=42):
    np.random.seed(seed)
    R_train, R_test = R.copy(), np.zeros_like(R)
    observed = list(zip(*np.where(R > 0)))
    np.random.shuffle(observed)
    for u, i in observed[:int(len(observed) * test_ratio)]:
        R_test[u, i] = R[u, i]
        R_train[u, i] = 0.0
    return R_train, R_test

In [18]:
# Summary
print(f"Users         : {R.shape[0]}")
print(f"Items         : {R.shape[1]}")
print(f"Ratings       : {np.count_nonzero(R)}")
print(f"Sparsity      : {100 * (1 - np.count_nonzero(R) / R.size):.2f}%")
R_train, R_test = train_test_split(R)
print(f"Train ratings : {np.count_nonzero(R_train)}")
print(f"Test ratings  : {np.count_nonzero(R_test)}")


Users         : 943
Items         : 1682
Ratings       : 100000
Sparsity      : 93.70%
Train ratings : 80000
Test ratings  : 20000


System of Equation Solver

In [19]:
# Print the SPD system of equations
np.random.seed(7)
k = 5
M = np.random.randn(k,k)
A = M.T @ M +  0.5 * np.eye(k)  # Make it SPD
b = np.random.randn(k)
print(f"A:\n{np.round(A, 2)}")
print(f"b:\n{np.round(b, 2)}")

x_true = np.linalg.solve(A, b)
print(f"True solution:\n{np.round(x_true, 2)}")




A:
[[ 8.59 -1.23 -1.08  3.81  0.96]
 [-1.23  1.08 -0.09  0.32 -0.44]
 [-1.08 -0.09  4.   -2.66 -1.37]
 [ 3.81  0.32 -2.66  5.96 -0.16]
 [ 0.96 -0.44 -1.37 -0.16  3.87]]
b:
[-1.45 -0.41 -2.29  1.05 -0.42]
True solution:
[-0.5  -1.22 -0.7   0.24 -0.36]


In [20]:
# Matrix Invserse Solver
def solve_inverse(A, b):
    A_inv = np.linalg.inv(A)
    x = A_inv @ b
    res = np.linalg.norm(A @ x - b)
    return x, res

x_inv, res_inv = solve_inverse(A, b)
print(f"Inverse solution:\n{np.round(x_inv, 2)}")
print(f"Residual norm: {res_inv:.2e}")


Inverse solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 5.12e-16


In [21]:
# Gaussian Elimination Solver
def solve_gaussian_elimination(A, b):
    k = len(b)
    M = np.hstack([A.astype(float).copy(), 
                    b.reshape(-1, 1).astype(float)])
    
    for col in range(k):
        # Partial pivoting
        max_row = col + np.argmax(np.abs(M[col:, col]))
        M[[col, max_row]] = M[[max_row, col]]

        pivot = M[col, col]
        for row in range(col + 1, k):
            factor = M[row, col] / pivot
            M[row,col:] -= factor * M[col, col:]

      # Back substitution
    x = np.zeros(k)
    for i in range(k - 1, -1, -1):
        x[i] = (M[i,k] - np.dot(M[i, i + 1:k], x[i + 1:k])) / M[i, i]
        res = float(np.linalg.norm(A @ x - b))
    return x, res

x_gauss, res_gauss = solve_gaussian_elimination(A, b)
print(f"Gaussian elimination solution:\n{np.round(x_gauss, 2)}")
print(f"Residual norm: {res_gauss:.2e}")

Gaussian elimination solution:
[-0.5  -1.22 -0.7   0.24 -0.36]
Residual norm: 5.55e-17
